# Trabalho de Modelos Lineares — Revisão para a Prova

**Estudos 1 e 2 | Python no Google Colab**

Este notebook acompanha os itens 1 a 7 do trabalho de Modelos Lineares, utilizando somente as análises estudadas em sala de aula.

**Objetivo:** realizar as análises estatísticas, compreender os resultados e preparar os gráficos e tabelas para o relatório final.

**Bases de dados:** os dois arquivos CSV devem ser colocados em `Meu Drive/Modelos_Lineares_IESB` antes da execução.

**Como utilizar:** executar as células na sequência apresentada. Os resultados serão salvos automaticamente em `Meu Drive/Modelos_Lineares_IESB/resultados`, em CSV, PNG de alta resolução e PDF.

**Importante:** o relatório final solicitado pela professora será entregue em PDF, sem códigos. Este notebook será utilizado para estudo e posterior publicação no GitHub.

## Preparação — conectar ao Drive e importar as bibliotecas
**Por quê?** Para ler as duas bases e salvar os resultados automaticamente, sem digitar novamente os dados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

# Caminho da pasta do trabalho no Google Drive

pasta = Path('/content/drive/MyDrive/Modelos_Lineares_IESB')

# Pasta onde serão salvos os resultados
saida = pasta / 'resultados'
saida.mkdir(exist_ok=True, parents=True)

# Configuração dos gráficos em alta resolução
plt.rcParams['savefig.dpi'] = 320

# Função para salvar gráficos em PNG e PDF
def salvar_grafico(nome):
    plt.tight_layout()
    plt.savefig(saida / f'{nome}.png', dpi=320, bbox_inches='tight')
    plt.savefig(saida / f'{nome}.pdf', bbox_inches='tight')
    plt.show()
    plt.close()

# Função para salvar tabelas em CSV, PNG e PDF
def salvar_tabela(tabela, nome):
    tabela.to_csv(saida / f'{nome}.csv', index=True)

    fig, ax = plt.subplots(
        figsize=(
            max(8, len(tabela.columns) * 1.5),
            max(2, len(tabela) * 0.45 + 1)
        )
    )

    ax.axis('off')
    texto = tabela.copy().round(4).astype(str)

    ax.table(
        cellText=texto.values,
        rowLabels=texto.index.astype(str),
        colLabels=texto.columns,
        loc='center',
        cellLoc='center'
    )

    salvar_grafico(nome)

# Verificação da pasta e dos arquivos
print("Pasta encontrada:", pasta.exists())
print("Arquivos CSV encontrados:")

for arquivo in pasta.glob('*.csv'):
    print("-", arquivo.name)

print("\nResultados serão salvos em:", saida)


## Base dos dois estudos
O **Estudo 1** tem temperatura (explicativa, em °C) e rendimento (resposta, em %). O **Estudo 2** tem dois fatores categóricos (Catalisador e Pressão), com duas repetições por combinação.

In [ ]:
# CARREGAMENTO DAS BASES DE DADOS

estudo1 = pd.read_csv(pasta / 'estudo1_processo_termico.csv')
estudo2 = pd.read_csv(pasta / 'estudo2_catalisador_pressao.csv')

print('Estudo 1:', estudo1.shape)
display(estudo1.head())

print('Estudo 2:', estudo2.shape)
display(estudo2.head())

## Parte I — Estudo 1
### Item 1 — Identificação e justificativa da abordagem
**Por quê?** Visualizamos se a produtividade acompanha uma reta ou uma curva. A subida seguida da queda aponta um modelo quadrático. Compararemos os graus 1, 2 e 3, como no material de aula.

In [ ]:
estudo1['Tc'] = estudo1['Temperatura'] - estudo1['Temperatura'].mean()
estudo1['Tc2'] = estudo1['Tc'] ** 2
estudo1['Tc3'] = estudo1['Tc'] ** 3
modelo1 = smf.ols('Rendimento ~ Tc', data=estudo1).fit()
modelo2 = smf.ols('Rendimento ~ Tc + Tc2', data=estudo1).fit()
modelo3 = smf.ols('Rendimento ~ Tc + Tc2 + Tc3', data=estudo1).fit()
temperaturas = np.linspace(estudo1.Temperatura.min(), estudo1.Temperatura.max(), 300)
grade = pd.DataFrame({'Tc': temperaturas - estudo1.Temperatura.mean()})
grade['Tc2'] = grade.Tc**2
grade['Tc3'] = grade.Tc**3
plt.figure(figsize=(8, 4.5))
plt.scatter(estudo1.Temperatura, estudo1.Rendimento, label='Observações')
for modelo, nome in [(modelo1,'Grau 1'), (modelo2,'Grau 2'), (modelo3,'Grau 3')]:
    plt.plot(temperaturas, modelo.predict(grade), label=nome)
plt.xlabel('Temperatura (°C)'); plt.ylabel('Rendimento (%)')
plt.legend(); plt.title('Estudo 1 — dispersão e modelos')
salvar_grafico('item_1_dispersao')

### Item 2.1 — Construção do modelo e centralização
**Por quê centralizar?** Os termos temperatura e temperatura² podem ficar muito correlacionados. Subtrair a média (Tc = Temperatura − média) facilita os cálculos e a interpretação, sem mudar as previsões.

In [ ]:
print('Temperatura média:', estudo1.Temperatura.mean())
print('Modelo quadrático (comparação):')
print(modelo2.summary())
print('Modelo cúbico (selecionado para discussão):')
print(modelo3.summary())
coeficientes = pd.DataFrame({'Estimativa':modelo3.params,'Erro-padrão':modelo3.bse,'t':modelo3.tvalues,'p-valor':modelo3.pvalues})
salvar_tabela(coeficientes, 'item_2_coeficientes')
print('Correlação sem centralização:', estudo1.Temperatura.corr(estudo1.Temperatura**2))
print('Correlação com centralização:', estudo1.Tc.corr(estudo1.Tc2))

### Item 2.2 — Comparação dos graus
**Por quê?** Um R² alto sozinho não prova que um modelo é o melhor. Comparamos R² ajustado, AIC/BIC e o teste F incremental. Preferimos o modelo mais simples que descreve bem o fenômeno. Neste conjunto, o termo cúbico é significativo, mas o teste de falta de ajuste ainda revela limitação; portanto, não declararemos o modelo perfeito.

In [ ]:
# ITEM 2 - COMPARAÇÃO DOS MODELOS POLINOMIAIS

# Comparação dos indicadores de qualidade
comparacao = pd.DataFrame({
    'R²': [m.rsquared for m in (modelo1, modelo2, modelo3)],
    'R² ajustado': [m.rsquared_adj for m in (modelo1, modelo2, modelo3)],
    'AIC': [m.aic for m in (modelo1, modelo2, modelo3)],
    'BIC': [m.bic for m in (modelo1, modelo2, modelo3)]
}, index=['Grau 1', 'Grau 2', 'Grau 3'])

salvar_tabela(comparacao, 'item_2_comparacao_graus')


# Teste F incremental - comparações diretas
f_1_2 = anova_lm(modelo1, modelo2)
f_2_3 = anova_lm(modelo2, modelo3)

print('Comparação: Grau 1 x Grau 2')
display(f_1_2)
salvar_tabela(f_1_2, 'item_2_teste_F_grau1_grau2')

print('Comparação: Grau 2 x Grau 3')
display(f_2_3)
salvar_tabela(f_2_3, 'item_2_teste_F_grau2_grau3')


# Temperatura de rendimento máximo - modelo cúbico
b0, b1, b2, b3 = modelo3.params

raizes = np.roots([3*b3, 2*b2, b1])

media = estudo1.Temperatura.mean()
limite_min = estudo1.Temperatura.min() - media
limite_max = estudo1.Temperatura.max() - media

candidatos = [
    r.real for r in raizes
    if abs(r.imag) < 1e-8
    and limite_min <= r.real <= limite_max
]

# Inclui as extremidades do intervalo estudado
candidatos += [limite_min, limite_max]

vertice_c = max(
    candidatos,
    key=lambda t: b0 + b1*t + b2*t**2 + b3*t**3
)

temperatura_otima = vertice_c + media
rendimento_otimo = (
    b0 + b1*vertice_c
    + b2*vertice_c**2
    + b3*vertice_c**3
)

print(f'Temperatura de máximo estimada: {temperatura_otima:.2f} °C')
print(f'Rendimento máximo estimado: {rendimento_otimo:.2f}%')

### Item 3.1 — Diagnóstico pelos resíduos
Para os diagnósticos, usamos o **grau 3**, que apresentou melhora estatisticamente significativa sobre o grau 2. **O que é resíduo?** É a diferença entre valor observado e previsto. O gráfico deve parecer uma nuvem sem desenho definido; o Q-Q ajuda a verificar normalidade e o gráfico de escala-localização, a variação dos erros.

In [ ]:
influencia = modelo3.get_influence()
estudo1['previsto'] = modelo3.fittedvalues
estudo1['residuo'] = modelo3.resid
estudo1['r_student'] = influencia.resid_studentized_internal
estudo1['alavancagem'] = influencia.hat_matrix_diag
estudo1['cook'] = influencia.cooks_distance[0]
plt.scatter(estudo1.previsto,estudo1.residuo)
plt.axhline(0,color='red');plt.xlabel('Ajustado');plt.ylabel('Resíduo');plt.title('Resíduos × ajustados')
salvar_grafico('item_3_residuos')
sm.qqplot(modelo3.resid,line='45',fit=True)
plt.title('Gráfico Q-Q dos resíduos');salvar_grafico('item_3_qq')
plt.scatter(estudo1.previsto,np.sqrt(np.abs(estudo1.r_student)))
plt.xlabel('Ajustado');plt.ylabel('Raiz de |resíduo studentizado|');plt.title('Escala-localização')
salvar_grafico('item_3_escala')

### Item 3.2 — Testes dos pressupostos

**Por quê?** Além dos gráficos, utilizamos testes estatísticos para avaliar se o modelo atende aos pressupostos da regressão.

- **Shapiro-Wilk:** verifica a normalidade dos resíduos.
- **Breusch-Pagan:** verifica se a variância dos erros é constante.

Adotamos o nível de significância de 5%. Quando o p-valor é menor que 0,05, rejeitamos a hipótese nula do teste. Quando é maior ou igual a 0,05, não encontramos evidências suficientes para rejeitá-la.

In [ ]:
# Teste de Shapiro-Wilk - Normalidade
w, p_shapiro = stats.shapiro(modelo3.resid)

# Teste de Breusch-Pagan - Homocedasticidade
bp = het_breuschpagan(modelo3.resid, modelo3.model.exog)

# Organização dos resultados
diagnostico = pd.DataFrame({
    'Teste': ['Shapiro-Wilk', 'Breusch-Pagan'],
    'Estatística': [w, bp[0]],
    'p-valor': [p_shapiro, bp[1]],
    'Conclusão': [
        'Normalidade não rejeitada' if p_shapiro >= 0.05
        else 'Evidência de não normalidade',

        'Variância constante não rejeitada' if bp[1] >= 0.05
        else 'Evidência de heterocedasticidade'
    ]
})

# Exibir tabela
display(diagnostico.round(4))

# Salvar tabela em alta resolução
salvar_tabela(diagnostico, 'item_3_testes_pressupostos')

# Durbin-Watson - Independência dos resíduos
dw = durbin_watson(modelo3.resid)

print(f'Durbin-Watson: {dw:.4f}')
print('Referência: valores próximos de 2 indicam')
print('ausência aproximada de autocorrelação de primeira ordem.')


### Item 3.3 — Pontos atípicos, alavancagem e influência
**Por quê?** Um ponto pode ter resíduo grande (atípico), temperatura extrema (alavancagem) ou alterar muito a equação (Cook). São coisas diferentes; não apagamos observações automaticamente.

In [ ]:
n = len(estudo1)
p = len(modelo3.params)

# Limites de referência estudados em sala
limite_alavancagem = 2 * p / n
limite_cook = 4 / n

# Gráfico de influência
plt.figure(figsize=(9, 6))

plt.scatter(
    estudo1.alavancagem,
    estudo1.r_student,
    s=100 + 2000 * estudo1.cook,
    alpha=0.6
)

plt.axvline(
    limite_alavancagem,
    color='red',
    linestyle='--',
    label='Limite de alavancagem'
)

plt.axhline(2, color='gray', linestyle=':')
plt.axhline(-2, color='gray', linestyle=':')

plt.xlabel('Alavancagem')
plt.ylabel('Resíduo studentizado')
plt.title('Influência (tamanho da bolha = Cook)')
plt.legend()

salvar_grafico('item_3_3_influencia')


# Identificação dos pontos que merecem atenção
estudo1['Atipico'] = abs(estudo1.r_student) > 2
estudo1['Alta_alavancagem'] = (
    estudo1.alavancagem > limite_alavancagem
)
estudo1['Possivel_influencia'] = (
    estudo1.cook > limite_cook
)

# Resumo dos resultados
print('Pontos atípicos:', estudo1.Atipico.sum())
print('Pontos com alta alavancagem:',
      estudo1.Alta_alavancagem.sum())
print('Pontos com Cook > 4/n:',
      estudo1.Possivel_influencia.sum())
print('Maior distância de Cook:',
      round(estudo1.cook.max(), 4))


# Tabela dos pontos que merecem investigação
pontos_atencao = estudo1[
    estudo1.Atipico |
    estudo1.Alta_alavancagem |
    estudo1.Possivel_influencia
][[
    'Temperatura',
    'Rendimento',
    'r_student',
    'alavancagem',
    'cook'
]].copy()

pontos_atencao.columns = [
    'Temperatura (°C)',
    'Rendimento (%)',
    'Resíduo studentizado',
    'Alavancagem',
    'Cook'
]

pontos_atencao = pontos_atencao.sort_values(
    'Temperatura (°C)'
)

print('\nObservações que merecem atenção:')
display(pontos_atencao.round(4))


# Salvar os resultados
pontos_atencao.round(4).to_csv(
    saida / 'item_3_3_pontos_atencao.csv',
    index=False
)

salvar_tabela(
    pontos_atencao,
    'item_3_3_pontos_atencao'
)

estudo1.to_csv(
    saida / 'item_3_diagnostico_completo.csv',
    index=False
)

### 3.4 Complemento visto em sala — falta de ajuste
Como há duas medições em cada temperatura, conseguimos separar o erro puro da falta de ajuste e comparar o modelo com as médias de cada temperatura.

In [ ]:
# Modelo com uma média para cada temperatura
modelo_medias = smf.ols(
    'Rendimento ~ C(Temperatura)',
    data=estudo1
).fit()

# ANOVA para verificar a falta de ajuste
falta_ajuste = anova_lm(modelo3, modelo_medias)

# Identificação dos modelos comparados
falta_ajuste.index = [
    'Modelo cúbico',
    'Modelo de médias'
]

# Nomes das colunas para facilitar a leitura
falta_ajuste = falta_ajuste.rename(columns={
    'df_resid': 'GL residual',
    'ssr': 'SQ residual',
    'df_diff': 'GL diferença',
    'ss_diff': 'SQ diferença',
    'F': 'F',
    'Pr(>F)': 'p-valor'
})

# Exibir e salvar a tabela

# Formatação da tabela para apresentação
tabela_falta_ajuste = falta_ajuste.round(4).fillna('—')

# Exibir a tabela organizada
display(tabela_falta_ajuste)

# Salvar em alta resolução
salvar_tabela(
    tabela_falta_ajuste,
    'item_3_4_falta_ajuste'
)

## Parte II — Estudo 2
### Item 4 — Estrutura do experimento
São **dois fatores fixos**: Catalisador (A, B) e Pressão (Baixa, Meia, Alta). São 2 × 3 = **6 combinações**, com duas medidas em cada. A resposta é o rendimento em %.

In [ ]:
# Estatísticas descritivas
resumo = estudo2.groupby(
    ['Catalisador', 'Pressao']
)['Rendimento'].agg(['count', 'mean', 'std']).reset_index()

# Renomear colunas
resumo.columns = [
    'Catalisador',
    'Pressão',
    'Quantidade',
    'Rendimento médio (%)',
    'Desvio-padrão'
]

# Organizar as pressões
resumo['Pressão'] = pd.Categorical(
    resumo['Pressão'],
    categories=['Baixa', 'Meia', 'Alta'],
    ordered=True
)

resumo = resumo.sort_values(
    ['Catalisador', 'Pressão']
).reset_index(drop=True)

# Exibir tabela
display(
    resumo.style
    .hide(axis='index')
    .format({
        'Rendimento médio (%)': '{:.2f}',
        'Desvio-padrão': '{:.2f}'
    })
)

# Salvar CSV sem numeração
resumo.to_csv(
    saida / 'item_4_descritivas.csv',
    index=False
)

# Salvar imagem da tabela sem índice
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

tabela = ax.table(
    cellText=resumo.round(2).values,
    colLabels=resumo.columns,
    loc='center',
    cellLoc='center'
)

tabela.auto_set_font_size(False)
tabela.set_fontsize(10)
tabela.scale(1, 1.5)

salvar_grafico('item_4_descritivas')

### Item 5 — ANOVA fatorial com interação
**Por quê?** Queremos saber se o Catalisador, a Pressão e a combinação dos dois alteram o rendimento. Um p-valor menor que 0,05 indica evidência de efeito estatisticamente significativo.

In [ ]:
# 1. Ajustar o modelo
modelo_fatorial = smf.ols(
    'Rendimento ~ C(Catalisador) * C(Pressao)',
    data=estudo2
).fit()

# 2. Tabela ANOVA
tabela_anova = anova_lm(modelo_fatorial, typ=2)

# Incluir o quadrado médio (QM)
tabela_anova['QM'] = (
    tabela_anova['sum_sq'] / tabela_anova['df']
)

# Organizar as colunas
tabela_anova = tabela_anova[
    ['df', 'sum_sq', 'QM', 'F', 'PR(>F)']
]

# Identificar os fatores
tabela_anova.index = [
    'Catalisador',
    'Pressão',
    'Catalisador × Pressão',
    'Resíduo'
]

# Renomear as colunas
tabela_anova.columns = [
    'GL', 'SQ', 'QM', 'F', 'p-valor'
]


# Formatação da ANOVA para apresentação
anova_formatada = tabela_anova.copy().round(4)

# Apresentar p-valores corretamente
anova_formatada['p-valor'] = tabela_anova['p-valor'].map(
    lambda x: '—' if pd.isna(x)
    else '< 0,001' if x < 0.001
    else f'{x:.4f}'
)

# Substituir valores ausentes por traços
anova_formatada = anova_formatada.fillna('—')

# Exibir e salvar
display(anova_formatada)

salvar_tabela(
    anova_formatada,
    'item_5_anova_fatorial'
)

# 3. Gráfico de interação entre os fatores
medias = estudo2.groupby(
    ['Pressao', 'Catalisador']
)['Rendimento'].mean().unstack()

medias = medias.reindex(['Baixa', 'Meia', 'Alta'])

plt.figure(figsize=(8, 5))
plt.plot(medias.index, medias['A'], marker='o', label='Catalisador A')
plt.plot(medias.index, medias['B'], marker='o', label='Catalisador B')

plt.xlabel('Pressão')
plt.ylabel('Rendimento médio (%)')
plt.title('Interação entre Catalisador e Pressão')
plt.legend()

salvar_grafico('item_5_interacao')


# 4. Resíduos do modelo
residuos2 = modelo_fatorial.resid

plt.figure(figsize=(8, 5))
plt.scatter(modelo_fatorial.fittedvalues, residuos2)
plt.axhline(0, color='red', linestyle='--')

plt.xlabel('Valores ajustados')
plt.ylabel('Resíduos')
plt.title('ANOVA - Resíduos x Valores Ajustados')

salvar_grafico('item_5_residuos')


# 5. Gráfico Q-Q
sm.qqplot(residuos2, fit=True, line='45')
plt.title('ANOVA - Gráfico Q-Q dos Resíduos')

salvar_grafico('item_5_qq')


# 6. Teste de normalidade - Shapiro-Wilk
w, p_shapiro2 = stats.shapiro(residuos2)

normalidade = pd.DataFrame({
    'Teste': ['Shapiro-Wilk'],
    'Estatística W': [w],
    'p-valor': [p_shapiro2],
    'Conclusão': [
        'Normalidade não rejeitada'
        if p_shapiro2 >= 0.05
        else 'Evidência de não normalidade'
    ]
})

display(normalidade.round(4))
salvar_tabela(normalidade, 'item_5_shapiro')


# 7. Desvio-padrão por combinação
desvios = estudo2.groupby(
    ['Catalisador', 'Pressao']
)['Rendimento'].std().reset_index()

desvios.columns = [
    'Catalisador', 'Pressão', 'Desvio-padrão'
]

display(desvios.round(4))

### Item 6 — Teste de Tukey (comparação entre combinações)
**Por quê?** O teste F mostra se existem diferenças; o Tukey revela quais pares diferem. Como pode haver interação, comparar as **seis combinações** permite escolher uma condição específica sem ignorar a interação.

In [ ]:
# 1. Criar as combinações de Catalisador e Pressão
estudo2['Combinacao'] = (
    estudo2['Catalisador'] + ' / ' + estudo2['Pressao']
)

# 2. Aplicar o teste de Tukey (5%)
tukey = pairwise_tukeyhsd(
    endog=estudo2['Rendimento'],
    groups=estudo2['Combinacao'],
    alpha=0.05
)

# 3. Organizar os resultados
tabela_tukey = pd.DataFrame(
    tukey._results_table.data[1:],
    columns=tukey._results_table.data[0]
)

tabela_tukey.columns = [
    'Grupo 1',
    'Grupo 2',
    'Diferença média',
    'p-valor ajustado',
    'Limite inferior',
    'Limite superior',
    'Diferença significativa'
]

# 4. Formatar a tabela para apresentação
tabela_formatada = tabela_tukey.copy()

for coluna in [
    'Diferença média',
    'Limite inferior',
    'Limite superior'
]:
    tabela_formatada[coluna] = tabela_formatada[coluna].map(
        lambda x: f'{x:.2f}'
    )

tabela_formatada['p-valor ajustado'] = (
    tabela_formatada['p-valor ajustado'].map(
        lambda x: '< 0,001' if x < 0.001 else f'{x:.4f}'
    )
)

tabela_formatada['Diferença significativa'] = (
    tabela_formatada['Diferença significativa'].map({
        True: 'Sim',
        False: 'Não'
    })
)

# 5. Exibir a tabela sem numeração
display(tabela_formatada.style.hide(axis='index'))

# 6. Salvar CSV sem índice
tabela_formatada.to_csv(
    saida / 'item_6_tukey.csv',
    index=False
)

# 7. Salvar tabela em PNG e PDF
fig, ax = plt.subplots(figsize=(15, 7))
ax.axis('off')

tabela = ax.table(
    cellText=tabela_formatada.values,
    colLabels=tabela_formatada.columns,
    loc='center',
    cellLoc='center'
)

tabela.auto_set_font_size(False)
tabela.set_fontsize(9)
tabela.scale(1, 1.5)

salvar_grafico('item_6_tukey')


# 8. Gráfico das médias por combinação
media_combinacao = estudo2.groupby(
    'Combinacao'
)['Rendimento'].mean().sort_values()

plt.figure(figsize=(9, 5))
media_combinacao.plot(kind='barh')

plt.xlabel('Rendimento médio (%)')
plt.ylabel('Combinação de Catalisador e Pressão')
plt.title('Rendimento médio por combinação')

salvar_grafico('item_6_medias')

## Parte III — Integração
### Item 7 — Relatório gerencial e conclusões


**Estudo 1 — Processo Térmico**

O modelo cúbico apresentou o melhor desempenho entre os graus avaliados, estimando uma temperatura de máximo rendimento de 205,68 °C, com rendimento previsto de 88,58%.

Entretanto, o teste de falta de ajuste foi significativo (p = 0,0217), indicando limitações na representação dos dados pelo modelo.

**Estudo 2 — Catalisador e Pressão**

A ANOVA identificou efeitos significativos dos dois fatores e da interação entre eles. O teste de Tukey indicou que a combinação Catalisador B + Pressão Alta apresentou o maior rendimento médio (84,55%), sendo significativamente superior às demais.

Contudo, o teste de Shapiro-Wilk identificou desvio da normalidade dos resíduos (p = 0,0169).

**Conclusão geral**

Os resultados sugerem utilizar uma temperatura próxima de 206 °C e priorizar a combinação Catalisador B com Pressão Alta.

Como os estudos foram realizados separadamente, essa combinação conjunta ainda precisa ser validada experimentalmente. Recomenda-se realizar novas medições antes de adotar definitivamente essas condições de produção.

## Exportação de todos os resultados
**Por quê?** Para guardar as figuras, tabelas e diagnósticos numa pasta compactada, sem perder qualidade ao copiar imagens da tela.

In [ ]:
import shutil
from google.colab import files

# Local onde o arquivo ZIP será salvo
destino_zip = pasta / 'Resultados_Modelos_Lineares'

# Compactar todos os arquivos da pasta resultados
arquivo_zip = shutil.make_archive(
    str(destino_zip),
    'zip',
    root_dir=saida
)

# Confirmar que o arquivo foi criado
print('Arquivo ZIP criado com sucesso!')
print('Local:', arquivo_zip)

# Baixar o ZIP para o computador
files.download(arquivo_zip)